# Gradient Descent for House Price Prediction

## Real-world problem
A real-estate team wants to estimate a property's sale price from its floor area, number of bedrooms, and age. We will train a **linear regression model** and use gradient descent to minimize the mean squared error.

**Dataset note:** This notebook generates a reproducible, simulated housing dataset with realistic relationships and random noise. It demonstrates the complete workflow without requiring an external download; it is not suitable for real property valuations.

### Learning objectives
- Translate a business problem into a cost function.
- Implement gradient descent for multiple features.
- Scale features so gradient descent converges reliably.
- Evaluate predictions using MAE, RMSE, and R².
- Compare the manual implementation with scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import SGDRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (8, 5)

### Create a sample housing dataset

The target is sale price in thousands of dollars. In this simulated example, larger homes generally cost more, extra bedrooms contribute to price, older homes receive a modest age-related discount, and random noise represents factors not captured by these features.

In [ ]:
n_samples = 600

area_sqft = np.random.uniform(600, 3500, n_samples)
bedrooms = np.clip(np.round(area_sqft / 650 + np.random.normal(0, 0.8, n_samples)), 1, 6)
age_years = np.random.uniform(0, 50, n_samples)

# Simulated price (in $1,000s), including noise.
noise = np.random.normal(0, 25, n_samples)
price_thousands = (
    45
    + 0.16 * area_sqft
    + 12 * bedrooms
    - 0.8 * age_years
    + noise
)

housing = pd.DataFrame({
    "area_sqft": area_sqft,
    "bedrooms": bedrooms.astype(int),
    "age_years": age_years,
    "price_thousands": price_thousands
})

display(housing.head())
print(f"Rows: {len(housing)}")
print(f"Average simulated price: ${housing['price_thousands'].mean():,.1f} thousand")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(housing["area_sqft"], housing["price_thousands"], alpha=0.55)
axes[0].set_xlabel("Floor area (sq ft)")
axes[0].set_ylabel("Price ($1,000s)")
axes[0].set_title("Price vs. floor area")

axes[1].hist(housing["price_thousands"], bins=25, edgecolor="black")
axes[1].set_xlabel("Price ($1,000s)")
axes[1].set_ylabel("Number of homes")
axes[1].set_title("Simulated price distribution")
plt.tight_layout()
plt.show()

### Prepare training and test data

We hold out test data to estimate how well the model predicts homes it has not seen. Standardization places features on comparable scales, which helps gradient descent take stable steps. The scaler is fit only on the training set to avoid data leakage.

In [ ]:
features = ["area_sqft", "bedrooms", "age_years"]
target = "price_thousands"

X = housing[features].to_numpy(dtype=float)
y = housing[target].to_numpy(dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add a column of ones for the intercept: prediction = intercept + X @ weights
X_train_b = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_b = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))
print("Feature order:", features)


## 3. Define the cost function and gradient

For $m$ training examples, linear regression predicts

$$
\hat{y} = X\theta
$$

We minimize mean squared error:

$$
J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}(\hat{y}_i-y_i)^2
$$

Its gradient is:

$$
\nabla J(\theta) = \frac{1}{m}X^T(X\theta-y)
$$

Gradient descent updates all parameters simultaneously:

$$
\theta := \theta - \alpha\nabla J(\theta)
$$

Here, $\alpha$ is the learning rate. The factor of 2 is omitted from the cost derivative by using the equivalent half-MSE objective.

In [ ]:
def compute_cost(X, y, theta):
    errors = X @ theta - y
    return np.mean(errors ** 2) / 2


def compute_gradient(X, y, theta):
    m = len(y)
    return (X.T @ (X @ theta - y)) / m


def gradient_descent(X, y, learning_rate=0.05, iterations=2000):
    theta = np.zeros(X.shape[1])
    cost_history = []

    for _ in range(iterations):
        theta -= learning_rate * compute_gradient(X, y, theta)
        cost_history.append(compute_cost(X, y, theta))

    return theta, cost_history


learning_rate = 0.05
iterations = 2000

theta, cost_history = gradient_descent(
    X_train_b, y_train, learning_rate=learning_rate, iterations=iterations
)

print("Learned parameters (scaled features):")
print(f"Intercept: {theta[0]:.2f} thousand dollars")
for name, weight in zip(features, theta[1:]):
    print(f"{name}: {weight:.2f} thousand dollars per standardized unit")
print(f"Final training cost (half-MSE): {cost_history[-1]:.3f}")

In [ ]:
plt.plot(cost_history)
plt.xlabel("Iteration")
plt.ylabel("Training cost (half-MSE)")
plt.title("Gradient descent convergence")
plt.grid(True, alpha=0.3)
plt.show()

### Evaluate performance on unseen homes

We use:
- **MAE:** average absolute prediction error, in thousands of dollars.
- **RMSE:** penalizes larger errors more heavily, in thousands of dollars.
- **R²:** proportion of target variance explained by the model; higher is better.

In [ ]:
y_pred = X_test_b @ theta

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  ${mae:.2f} thousand")
print(f"Test RMSE: ${rmse:.2f} thousand")
print(f"Test R²:   {r2:.3f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.7)
lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual price ($1,000s)")
plt.ylabel("Predicted price ($1,000s)")
plt.title("Actual vs. predicted house prices")
plt.grid(True, alpha=0.3)
plt.show()

### Try a practical prediction

Enter the characteristics of a hypothetical property. The prediction is in thousands of dollars and is based only on this notebook's simulated data.

In [ ]:
def predict_house_price(area_sqft, bedrooms, age_years):
    new_home = np.array([[area_sqft, bedrooms, age_years]], dtype=float)
    new_home_scaled = scaler.transform(new_home)
    new_home_b = np.c_[np.ones(new_home_scaled.shape[0]), new_home_scaled]
    prediction_thousands = float((new_home_b @ theta).item())
    return prediction_thousands


example_area = 1800
example_bedrooms = 3
example_age = 10


predicted_price = predict_house_price(example_area, example_bedrooms, example_age)

print("Hypothetical property")
print(f"Area: {example_area:,} sq ft")
print(f"Bedrooms: {example_bedrooms}")
print(f"Age: {example_age} years")
print(f"Estimated price: ${predicted_price:,.1f} thousand")

### Compare with scikit-learn's SGDRegressor

`SGDRegressor` provides a library implementation of stochastic gradient descent. It is not expected to match the manual batch-gradient-descent coefficients exactly because the optimization procedure and stopping criteria differ, but the test metrics should be in a comparable range.

In [ ]:
library_model = SGDRegressor(
    loss="squared_error",
    penalty=None,
    max_iter=5000,
    tol=1e-5,
    learning_rate="constant",
    eta0=0.005,
    random_state=RANDOM_STATE
)
library_model.fit(X_train_scaled, y_train)

library_pred = library_model.predict(X_test_scaled)

comparison = pd.DataFrame({
    "Model": ["Manual batch gradient descent", "scikit-learn SGDRegressor"],
    "MAE ($1,000s)": [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, library_pred)
    ],
    "RMSE ($1,000s)": [
        np.sqrt(mean_squared_error(y_test, y_pred)),
        np.sqrt(mean_squared_error(y_test, library_pred))
    ],
    "R²": [
        r2_score(y_test, y_pred),
        r2_score(y_test, library_pred)
    ]
})

display(comparison.round(3))

### interpretation and limitations

- Gradient descent learns parameters that reduce prediction error on the training data.
- The held-out metrics provide a basic check of generalization.
- Feature scaling is important because floor area, bedroom count, and age use different numeric ranges.
- This dataset is simulated. Real deployment requires verified transaction data, location and property-condition features, careful validation, and monitoring for changing market conditions.
- A low error on this example does not establish that the model is suitable for actual buying, selling, lending, or valuation decisions.

### Key takeaway
Gradient descent is useful when a model's parameters can be optimized by following the negative gradient of a loss function. Here, it turns housing features into a price estimate by minimizing prediction error.